In [1]:
import sys

sys.path.insert(0, '../../')

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import yaml
from jax import vmap

from iactrace import MCIntegrator, Telescope
from iactrace.viz import show_telescope

# Some useful code:

In [2]:
def spherical_surface(radius, points):
    Z = radius-np.sqrt(radius**2-(points[...,0]**2+points[...,1]**2))
    N = np.stack([-points[...,0], -points[...,1], radius - Z], axis=-1)
    return np.stack([points[...,0], points[...,1], Z], axis=-1), N/radius

def parabolic_surface(focal_length, points):
    a = 1.0 / (4.0 * focal_length)
    Z = a * (points[..., 0]**2 + points[..., 1]**2)
    ones = jnp.ones_like(Z)
    N = jnp.stack([-2*a*points[..., 0], -2*a*points[..., 1], ones], axis=-1)
    norm = jnp.sqrt(4*a*Z + 1)
    positions = jnp.stack([points[..., 0], points[..., 1], Z], axis=-1)

    return positions, N / norm[..., None]

def look_at_euler(mirror_pos, target_pos, up=None):
    """
    Compute euler angles to look from mirror_pos towards target_pos.

    Args:
        mirror_pos: Position to look from (3,)
        target_pos: Position to look at (3,)
        up: Up vector (3,)

    Returns:
        Euler angles (3, )
    """
    if up is None:
        up = jnp.array([0., 1., 0.])
    forward = target_pos - mirror_pos
    forward = forward / jnp.linalg.norm(forward)

    right = jnp.cross(up, forward)
    right = right / jnp.linalg.norm(right)

    up_corrected = jnp.cross(forward, right)

    # Rotation matrix: local -> world
    R = jnp.column_stack([right, up_corrected, forward])

    # Compute tilt
    sy = -R[2,0]  # sin(tilt)
    cy = jnp.sqrt(1 - sy**2)

    # Avoid gimbal problem
    tip = jax.lax.cond(
        cy > 1e-6,
        lambda _: jnp.arctan2(R[2,1], R[2,2]),
        lambda _: 0.0,
        operand=None
    )
    rotation = jax.lax.cond(
        cy > 1e-6,
        lambda _: jnp.arctan2(R[1,0], R[0,0]),
        lambda _: jnp.arctan2(-R[0,1], R[1,1]),
        operand=None
    )
    tilt = jnp.arcsin(sy)

    return jnp.rad2deg(jnp.array([tip, tilt, rotation]))

# Loading config files:

Loading files:

In [3]:
# Position of hexagon pixels in LSTCAM:
lstpix = np.load('lst_pix.npy')

In [86]:
# Get filenames for mirror positions:
mirror_files = ['mirror_CTA-N-LST1_v2019-03-31_rotated.dat',
                'mirror_CTA-N-LST2_v2020-04-07_rotated.dat',
                'mirror_CTA-N-LST3_v2020-04-07_rotated.dat',
                'mirror_CTA-N-LST4_v2020-04-07_rotated.dat'
               ]

# Create 4 northern LSTs with individual mirror files:

In [88]:
for i in range(4):
    df = pd.read_csv(mirror_files[i], comment='#', sep=r'\s+',
                 names=['x', 'y', 'd', 'f', 'w', 't'])
    df = df.apply(lambda x: x.astype(str).str.replace(',', ''))
    df = df.astype(float)

    mirrors_xydf = np.array(df[['x', 'y', 'd', 'f']].values)/100

    t_name = f"LST_{i+1}_North_like"

    # Focal length
    f = 28

    # Build config dictionary directly
    config = {
        "telescope": {"name": t_name, "units": "m"},
        "mirror_templates": {
            "spherical_base": {
                "surface": {
                    "curvature": 1/(2*28.5),
                    "conic": 0.0,
                    "aspheric": [],
                }
            }
        },
        "mirrors": [],
        "obstructions": [],
        "sensors": [],
    }

    # Add hexagonal sensor (LSTCAM)
    config["sensors"].append({
        "id": "LSTCAM",
        "type": "hexagonal",
        "position": [0.0, 0.0, f],
        "orientation": [0.0, 0.0, 0.0],
        "centers_x": lstpix[:,0].tolist(),
        "centers_y": lstpix[:,1].tolist(),
        "edge_width": 0.001,
    })

    ## Calculate intersection point for each mirror point
    z_val = 2*f + np.sum(mirrors_xydf[:,:2]**2, axis=1) / (4*f)
    p_orient = jnp.stack([jnp.zeros_like(z_val), jnp.zeros_like(z_val), z_val]).T

    ## For all mirrors, calculate position and rotation:
    Tmirrors, _ = parabolic_surface(36, mirrors_xydf[:,:2])
    Rmirrors = np.array(vmap(look_at_euler, in_axes= (0, 0))(Tmirrors, p_orient))
    Rmirrors[:,2] = 0

    ## Define hexagon vertices
    D = 1.51  # flat-to-flat distance in meters
    R = D / np.sqrt(3)  # distance from center to vertex

    # angles for vertices
    angles = np.deg2rad(np.arange(0, 360, 60))

    # coordinates of vertices
    vertices = [[float(R * np.cos(a)), float(R * np.sin(a))] for a in angles]

    ## Add camera obstruction
    config["obstructions"].append({
        "id": "Camera_Enclosure",
        "type": "box",
        "p1": [-1.57, -1.57, 27.5],
        "p2": [1.57, 1.57, 29.0],
    })

    ## Add camera holding structure
    def ellipse(r, a):
        return 28.5 * (1 - (r / a) ** 2) ** 0.5

    def get_arc_length_points(a, start, n_points, n_samples=1000):
        r_samples = np.linspace(start, a, n_samples)
        ds = np.sqrt(np.diff(r_samples)**2 + np.diff(ellipse(r_samples, a))**2)
        s = np.concatenate([[0], np.cumsum(ds)])
        return np.interp(np.linspace(0, s[-1], n_points), s, r_samples)

    # Mast structure
    N_seg = 20
    for mid, a, sign in [(0, 11.5, 1), (1, 12.5, -1)]:
        for j in range(N_seg):
            r1 = 1.57 + j * (a - 1.57) / N_seg
            r2 = 1.57 + (j + 1) * (a - 1.57) / N_seg
            config["obstructions"].append({
                "id": f"Mast_{mid}_{j}",
                "type": "cylinder",
                "p1": [float(sign * r1), 0.0, float(ellipse(r1, a))],
                "p2": [float(sign * r2), 0.0, float(ellipse(r2, a))],
                "r": 0.155,
            })

    # Camera wires
    for side, y in [("L", -1), ("R", 1)]:
        config["obstructions"].append({
            "id": f"Wire_{side}_Camera",
            "type": "cylinder",
            "p1": [0.0, float(y * 11.5), 1.5],
            "p2": [0.0, float(y * 1.56), 27.8],
            "r": 0.01,
        })

    # Tension wires
    for mid, a, sign in [(0, 11.5, 1), (1, 12.5, -1)]:
        r_pts = get_arc_length_points(a, 2.5, 8)
        for j in range(6):
            x_base = sign if j > 1 else 0
            for side, y in [("L", -1), ("R", 1)]:
                config["obstructions"].append({
                    "id": f"Wire_{side}_{mid}_{j}",
                    "type": "cylinder",
                    "p1": [float(x_base), float(y * 11.5), 1.5],
                    "p2": [float(sign * r_pts[j]), 0.0, float(ellipse(r_pts[j], a))],
                    "r": 0.01,
                })

    ## Add mirrors one by one
    for j in range(len(Tmirrors)):
        config["mirrors"].append({
            "id": f"M_{j}",
            "template": "spherical_base",
            "position": [float(x) for x in Tmirrors[j]],
            "orientation": [float(x) for x in Rmirrors[j]],
            "aperture": {
                "type": "polygon",
                "vertices": vertices,
            },
            "curvature": 1.0 / (2.0 * mirrors_xydf[j,-1]),
            "stage": 0,
        })

    # Save config to YAML file
    with open(t_name + '.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)

In [89]:
telescope = Telescope.from_yaml('LST_1_North_like.yaml', MCIntegrator(256), key = jax.random.key(42))
scene = show_telescope(telescope)
scene.show(viewer='jupyter')